In [1]:
from pathlib import Path
import pandas as pd

data_dir = Path("../data/raw/ieee-fraud-detection")

df_trans = pd.read_csv(data_dir / "train_transaction.csv")
df_id = pd.read_csv(data_dir / "train_identity.csv")

In [2]:
print("TransactionID unique in trans:", df_trans['TransactionID'].is_unique)
print("TransactionID unique in identity:", df_id['TransactionID'].is_unique)

TransactionID unique in trans: True
TransactionID unique in identity: True


In [36]:
match_rate = df_trans['TransactionID'].isin(df_id['TransactionID']).mean() * 100
print(f"Matching identity record: {match_rate:.2f}%")

Matching identity record: 24.42%


In [4]:
# Merge the two tables using Left Join
df_merged = df_trans.merge(df_id, on='TransactionID', how='left')

In [5]:
# Compare fraud rate: With Identity vs Without Identity
df_merged['has_identity'] = df_merged['DeviceType'].notna()
print(df_merged.groupby('has_identity')['isFraud'].agg(['count', 'mean']))

               count      mean
has_identity                  
False         449730  0.021017
True          140810  0.079618


/tmp/ipykernel_8365/2128245842.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['has_identity'] = df_merged['DeviceType'].notna()


In [6]:
# Fraud rate by DeviceType (Mobile vs Desktop)
print(df_merged.groupby('DeviceType')['isFraud'].agg(['count', 'mean']))

            count      mean
DeviceType                 
desktop     85165  0.065215
mobile      55645  0.101662


In [37]:
selected_cols = [
    # # Identifiers & Target
    # 'TransactionID', 'isFraud', 'TransactionDT',
    # # Core Transaction Metrics
    # 'TransactionAmt', 'ProductCD',
    # # Card
    # 'card1', 'card2', 'card4', 'card6',
    # # Address
    # 'addr1', 'addr2', 'dist1',
    # # Email Domains
    # 'P_emaildomain', 'R_emaildomain',
    # Identity features
    'DeviceType', 'DeviceInfo', 'id_30', 'id_31'
]

In [38]:
df_dw_ready = df_merged[selected_cols]

In [39]:
df_dw_ready.head()

,DeviceType,DeviceInfo,id_30,id_31
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN
4,mobile,SAMSUNG SM-G892A Build/NRD90M,Android 7.0,samsung browser 6.2
